# Ledger: Expenses 

This notebook manages tge general ledger for the LLC


## Status:

- Import : Bk -> ExpDb
    - create 
    - update New only
    - save
    - load
    - data fields 
- Exp Editor 
    - add purchase orders
    - reconcile Bk expenses
    - delete
- Report -> Exp Leger
    - Monthly Report
    - YE Report






In [5]:
# Load bookkeeping services
import os
from ledger.LLC import LLC
from pathlib import Path
import datetime
import pandas as pd
from IPython.display import display, Markdown

top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=False, top=top)

# 
dtReport = datetime.datetime.now().strftime('%Y.%m.%d')

display(Markdown(f"### Profile - {dtReport}"))
display(Markdown(f"- **LLC Name: {llc.objName}**"))
display(Markdown(f"- **Year: {llc.yr}**"))



### Profile - 2026.03.16

- **LLC Name: WBGroupLLC**

- **Year: 2025**

## General Ledger: Incomes/Expenses

In [6]:
# General Ledger - All accounts
llc._Bank()
glDF = llc.bk.df.groupby(['Acct']).amt.sum()
glDF.loc[f'Balance'] = glDF.sum()
print(glDF.to_string())

Acct
Acct.Asset.Purchase    -214113.95
Acct.Cash.Expense        -1766.92
Acct.Cash.Income          4000.53
Acct.Cash.Investment    219227.00
Acct.Cash.Misc              29.47
Acct.Cash.Util           -1056.95
Acct.Interest.Income       400.00
Balance                   6719.18


## Expense Ledger

- expense ledger  DB : `AccountingData/Accts/llc<name>_expenses.json
- import new Bank expenses into exp ledger
- Reconcile expLedger to receipts
- gen ExpenseLedger report
- import expense details to tax forms

## Ledger - Expenses Details

In [7]:
llc.bk.df.head(4)

,dt,amt,C2,CheckNo,desc,TransType,Acct,AcctSub,TDesc
0,12/29/2025,-177.0,*,NaN,Cash eWithdrawal in Branch 12/29/2025 13:47 PM...,Exp,Acct.Asset.Purchase,p20251229-RV1,Purchase Rental RV
1,12/29/2025,177.0,*,NaN,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...,Rev,Acct.Cash.Investment,o20250801_1,Initial Investment by member
2,12/26/2025,-135.8,*,NaN,ALLSTATE IND CO INS PYMT DEC024 00000043853221...,Exp,Acct.Cash.Util,Ins_Home,Pay Monthly Util
3,12/18/2025,-50.0,*,NaN,BILL PAY Water-COMWSC RECURRING 38 ON 12-18,Exp,Acct.Cash.Util,Water,Pay Monthly Util


In [9]:
## Expenses: Bank Summary

In [8]:
df3 = llc.bk.df
df3 = df3[df3.TransType == 'Exp']
pd.DataFrame(df3.groupby(['Acct','AcctSub']).amt.sum())

amt
Acct                AcctSub                    
Acct.Asset.Purchase p20250826-805HMD -213936.95
                    p20251229-RV1       -177.00
Acct.Cash.Expense   Maintenance         -135.00
                    amazon              -235.49
                    h-e-b               -173.68
                    harbor               -60.58
                    hays                 -32.00
                    kings                -25.94
                    laird               -487.13
                    lowe's               -27.04
                    lowes                -31.86
                    sp                   -74.57
                    sq                   -57.31
                    texas                -90.90
                    wal-mart            -140.73
                    wimberley           -235.79
Acct.Cash.Misc      Misc                  -0.53
Acct.Cash.Util      Elec                -571.15
                    Ins_Home            -135.80
                    Util                 -50.00
                    Water               -300.00

In [ ]:
import os
from ledger.ledgerObject import ledgerObject
import pandas as pd
import datetime


# class llcAssets
class llcExpenses(ledgerObject):
    def __init__(self, llc, **kwargs):
        super().__init__(llc, **kwargs)
        if self.debug: print(f"llc:{self.oID} {type(self).__name__} Init Done")
            
    def FN(self): 
        fn = os.path.join(self.llc.TOP, self.llc.dirAccounting, 'Accts', f"{self.oID}_{self.llc.objName}.json")
        if self.debug: print(f"{self.oID} ledgerObject.FN: {fn}")
        return fn

    def reconcile(self, df):
        # Check if Bk df has new transactions
        eDF = pd.DataFrame(self.load())
        eIDList = list(eDF.eID)
        bkS = df.apply(lambda r, self._expKey(r) in eDF.eID, axis=1)
        

    def _expKey(self, r):
        return f"{r.dt}_{r.amt:0.2f}_{r.Acct}_{r.AcctSub}"

    def _payee(self, r):
        '''
        Use Acct and AcctSub as default,   reconcilation can change this
        '''
        return r.AcctSub

    def catDict(self, r, cat=None, ):
        '''
        Common Expense Categories
        '''
        expCatDict = dict(maint  = [],
                          maint_CLN = [],
                          util = [],
                          ins = [],
                          admin = [], # Mgmt, Avertizing, Legal, 
                          morg = [],
                          misc = [] # Travel, Office, etc...
                         )
        if cat is None: 
            return expCatDict.keys()

        
        try:
            expCatDict[cat].append(r.eID)
        except:
            # Initialize category; add exp ID to cross ref
            expCatDict[cat] = [r.eID]
        return cat
        

    def newExpDict(self, r):
                                                                            

        eDict = dict(eID = self._expKey(r),
                     dt = r.dt,
                     amt = r.amt,
                     Acct = r.Acct,
                     AcctSub = r.AcctSub,
                     desc = r.Tdesc,
                     category =  'misc', # IRS Schedule E category.
                     payee = self._payee(r),
                     payMethod = 'cc',
                     assetID = None,
                     notes = '',
                     receipt = None,  # None, Y, N, Bk
                     reconciled = dict(o=None, dt=None)
                    )
                                                                               asada
                                                                               
        

    def bk2Exp(self, df):
        '''
        Import Bk DF and produce a set of new expense entries
          - assign unique ID to each Expense:  dt_amt_Acct_Acct_Sub
          - Expense Recored
                Date: The date the expense was paid.
                Description: A clear description of the item or service (e.g., "HVAC repair," "Monthly Landscaping").
                Amount: The total cost.
                Category: The IRS Schedule E category.
                Vendor/Payee: Who was paid.
                Payment Method: Check #, Credit Card, or Bank Transfer.
                Supporting Documentation: Receipt or invoice number. 
        '''

        df == df[df.TransType == 'Exp'].copy()

    
        df3 = df3[df3.TransType == 'Exp']
        pd.DataFrame(df3.groupby(['Acct','AcctSub']).amt.sum())
        
        
    """
    -----------------------------------------------
    Services for classification and reconcilation
    - call .fetch() before using services
    -----------------------------------------------
    """
    def fetch(self):
        '''
        Load assets into df
        Load aDict to get keys to match
        '''
        try:
            len(self.df)
        except:
            self.df = pd.DataFrame(self.load())
            self.kDict = self._keyDict()

    def _key(self, r):
        # Return tuple of k,index for given row (dt, amt)
        # Return a unique key id for asset entry
        return (f"{r['dt']}_{r.amt}",r.name)
    
    def _keyDict(self):
        # map dt :: amt, return dict dt:amt 
        df = self.df
        df['dt'] = df.dt.apply(lambda v: datetime.datetime.strptime(v, '%Y.%m.%d').strftime('%m/%d/%Y'))

        kList = df.apply(lambda r : self._key(r), axis=1)
        return {r[0]:r[1] for r in kList}
    
    def _matchBk(self, r):
        # Classify transation
        # Match Bank transaction to asset event based on date(dt) and amount(amt)
        # Return tuple of Acct, SubAcct, and Desc
        (bkKey,ndx) = self._key(r)
        try:
            ndx = self.kDict[bkKey]
            aRow = self.df.iloc[ndx]
            subAcct = aRow.oID
            subAcct = subAcct if subAcct[0] == 'p' else ','.join(aRow.stakeholderPct.keys())
            
            return (aRow.acct, subAcct, aRow.desc)
        except:
            return None
        

